# Extracting data from Resumes

Let us assume that we are running a hiring process for a company and we have received a list of resumes from candidates. We want to extract structured data from the resumes so that we can run a screening process and shortlist candidates. 

Take a look at one of the resumes in the `data/resumes` directory. 

In [ ]:
from dotenv import load_dotenv
from llama_cloud_services import LlamaExtract
from pydantic import BaseModel, Field
import os
import asyncio
from pathlib import Path
from llama_index.core import Document
from typing import List
from llama_index.core.prompts import PromptTemplate
from llama_index.core.async_utils import run_jobs
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface_api import HuggingFaceInferenceAPIEmbedding
from llama_index.core import Document
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
import json
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


# Load environment variables (put LLAMA_CLOUD_API_KEY in your .env file)
load_dotenv(dotenv_path=r"C:\Users\ogida\Desktop\Hope work\Tech\Vscode_files\ScaleSense\.env")

# Optionally, add your project id/organization id
llama_extract = LlamaExtract(api_key=os.getenv("LAMMA_CLOUD_API_KEY"))

c:\Users\ogida\Desktop\Hope work\Tech\Vscode_files\ScaleSense\scale\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# from llama_cloud.core.api_error import ApiError

# try:
#     existing_agent = llama_extract.get_agent(name="resume-screening")
#     # if existing_agent:
#     #     llama_extract.delete_agent(existing_agent.id)
# except ApiError as e:
#     if e.status_code == 404:
#         pass
#     else:
#         raise

# # agent = llama_extract.create_agent(name="resume-screening", data_schema=Resume)

In [3]:
llama_extract.list_agents()

[ExtractionAgent(id=f9cdcdb6-1372-4fcd-aa35-cd1a23e540be, name=resume-screening)]

In [ ]:
from typing import List, Optional


class Education(BaseModel):
    institution: str = Field(description="The institution of the candidate")
    degree: str = Field(description="The degree of the candidate")
    start_date: Optional[str] = Field(
        default=None, description="The start date of the candidate's education"
    )
    end_date: Optional[str] = Field(
        default=None, description="The end date of the candidate's education"
    )


class Experience(BaseModel):
    company: str = Field(description="The name of the company")
    title: str = Field(description="The title of the candidate")
    description: Optional[str] = Field(
        default=None, description="The description of the candidate's experience"
    )
    start_date: Optional[str] = Field(
        default=None, description="The start date of the candidate's experience"
    )
    end_date: Optional[str] = Field(
        default=None, description="The end date of the candidate's experience"
    )


# class Resume(BaseModel):
#     name: Optional[str] = Field(description="The name of the candidate")
#     email: Optional[str] = Field(description="The email address of the candidate")
#     links: Optional[List[str]] = Field(
#         default=None, description="The links to the candidate's social media profiles"
#     )
#     experience: List[Experience] = Field(description="The candidate's experience")
#     education: List[Education] = Field(description="The candidate's education")



class TechnicalSkills(BaseModel):
    programming_languages: List[str] = Field(
        description="The programming languages the candidate is proficient in."
    )
    frameworks: List[str] = Field(
        description="The tools/frameworks the candidate is proficient in, e.g. React, Django, PyTorch, etc."
    )
    skills: List[str] = Field(
        description="Other general skills the candidate is proficient in, e.g. Data Engineering, Machine Learning, etc."
    )


class Resume(BaseModel):
    name:  Optional[str] = Field(description="The name of the candidate")
    email:  Optional[str] = Field(description="The email address of the candidate")
    links: List[str] = Field(
        description="The links to the candidate's social media profiles"
    )
    country: Optional[str] = Field(
        default=None, description="The country the candidate is based in, if available"
    )
    experience: List[Experience] = Field(description="The candidate's experience")
    education: List[Education] = Field(description="The candidate's education")
    technical_skills: TechnicalSkills = Field(
        description="The candidate's technical skills"
    )
    key_accomplishments: str = Field(
        description="Summarize the candidates highest achievements."
    )
    years_of_experience: int = Field(
        description="The total years of experience the candidate has based on the number of years in each experience entry, if available"
    )
    domain: Optional[str] = Field(
        default=None, description="The domain the candidate has experience in, e.g. Finance, Healthcare, etc."
    )
    file_path: Optional[str] = Field(
        default=None, description="The file path of the resume that was parsed"
    )


# agent = llama_extract.create_agent(name="resume-screening", data_schema=Resume)
    
# agent.data_schema = Resume
      

In [ ]:
# agent.save()

In [6]:
agent = llama_extract.get_agent("resume-screening")
# agent.data_schema  # Latest schema should be returned

In [7]:


raw_resume_data = []


resumes = []
with os.scandir(r"C:\Users\ogida\Desktop\Hope work\Tech\Vscode_files\ScaleSense\src\data\sampled_data\IT") as entries:
    for entry in entries:
        if entry.is_file():
            resumes.append(entry.path)

# 1. Queue the extraction
jobs = await agent.queue_extraction(resumes)

documents_for_chroma = []

# 2. Zip the original file paths and the resulting jobs together
for file_path, job in zip(resumes, jobs):
    
    # Poll the API until the extraction is finished
    while True:
        job_status = agent.get_extraction_job(job.id).status
        if job_status == "SUCCESS":
            break
        elif job_status in ["FAILED", "CANCELLED"]:
            print(f"Extraction failed for: {file_path}")
            break

        await asyncio.sleep(2) # Wait 2 seconds before checking again
        
    # 3. Process the successful result
    if job_status == "SUCCESS":
        result = agent.get_extraction_run_for_job(job.id)
        
        # Extract the data (Convert from Pydantic to dict if necessary)
        extracted_data = result.data 

        extracted_data['file_path'] = file_path  # Add the file path to the extracted data

        raw_resume_data.append(extracted_data)

Creating extraction jobs: 100%|██████████| 10/10 [00:01<00:00,  6.08it/s]


## Indexing the extracted resume

In [8]:
# raw_resume_data[0]

# 1. Initialize your ChromaDB Client and Vector Store
db = chromadb.PersistentClient(path="../chroma_db")
chroma_collection = db.get_or_create_collection("parsed_documents")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)



# hf_embed_model = HuggingFaceInferenceAPIEmbedding(
#     model_name="BAAI/bge-large-en-v1.5", 
#     token=os.getenv("HUGGING_FACE_HUB_TOKEN") # Pulls your API token securely
# )

In [9]:

local_embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# 3. Define the Transformations for the Pipeline
transformations = [
    SentenceSplitter(chunk_size=512, chunk_overlap=50),
    local_embed_model
]

# 4. Create the Ingestion Pipeline
pipeline = IngestionPipeline(
    transformations=transformations,
    vector_store=vector_store
)

# 5. Prepare your Documents
documents_to_process = []

c:\Users\ogida\Desktop\Hope work\Tech\Vscode_files\ScaleSense\scale\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ogida\AppData\Local\llama_index\llama_index\Cache\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103

In [10]:
# raw_resume_data[0]

In [11]:
for data in raw_resume_data: 
    technical_skills = data.get('technical_skills', {})
    skills = technical_skills.get('skills', []) if technical_skills else []
    skills_string = ", ".join(skills)
    
    metadata_dict = {
        'skills': skills_string,
        'country': data.get('country', 'Unknown'),
        'domain': data.get('domain', 'Unknown'),
        'years_of_experience': data.get('years_of_experience', 0),
        'file_path': data.get('file_path', '')
    }

    text_payload = json.dumps(data, indent=2)
    
    doc = Document(
        text=text_payload,
        metadata=metadata_dict,
        excluded_llm_metadata_keys=['file_path']
    )
    documents_to_process.append(doc)

print(f"Prepared {len(documents_to_process)} documents for ingestion.")

# 6. Run the Pipeline
# Note: Because this makes network calls to an API, having num_workers > 1 
# is very helpful for speeding up the batch processing!
nodes = pipeline.run(
    documents=documents_to_process, 
    show_progress=True,
    num_workers=4 
)

print(f"✅ Successfully processed and saved {len(nodes)} chunks into ChromaDB!")

Prepared 10 documents for ingestion.


✅ Successfully processed and saved 61 chunks into ChromaDB!
